# M26 — Diagnose Deep Learning Failure

**Objective:** debug optimization, data, and architecture failures systematically.

M25 built a CPU training loop

`zero_grad → forward → loss → backward → step`

with protected splits, eval/no-grad, and checkpoints. M26 asks **why a
run that still emits numbers is the wrong run**. The useful whole is the
known-good M25 trace. Then one fault at a time. Then a hidden Chaos Day.

Language-model internals stay closed (P5). A running notebook is not
competence.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a sign on the loss, a split size, a
`requires_grad` flag, a feature-scale ratio, or whether parameters move.

CPU is canonical. Do not download weights, do not require a GPU, and do
not treat a decreasing loss as proof that the run is healthy. If the
failure is in labels, scaling, learning rate, a blocked path, capacity,
Dropout, or evaluation, stay there.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M26" / "dl_failure_lab.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M25.pytorch_training import (
    LOOP_ORDER,
    TRAIN_HIDDEN,
    compact_run_report,
    train_model,
)
from missions.M26.dl_failure_lab import (
    CHAOS_SEED,
    DEFAULT_SEED,
    HIDDEN_PRACTICE_SEED,
    apply_model_hooks,
    claimed_eval,
    cheapest_discriminator,
    chaos_day,
    cluster_label_agreement,
    diagnostic_battery,
    empty_diagnosis_record,
    feature_column_scales,
    feature_scale_ratio,
    healthy_invariants_hold,
    hidden_fault_run,
    invariant_failures,
    known_good_m25_trace,
    m25_loop_order,
    prepare_fault,
    public_symptoms,
    rank_hypotheses,
    repair_and_verify,
    repair_prepared,
    run_prepared,
    signals_from_battery,
    tiny_subset_overfit_check,
)

print("repository root:", ROOT)
print("M25 loop order:", LOOP_ORDER, m25_loop_order())
print("train hidden:", TRAIN_HIDDEN)
print("diagnosis record keys:", sorted(empty_diagnosis_record()))
print("train_model is the M25 runner:", train_model.__module__)


## M25 boundary: keep the instrumented loop

M26 may break data, optimization, architecture, or evaluation **only
after** the M25 loop is the runner. Import
`missions.M25.pytorch_training.train_model`. Do not rewrite
`training_step`. If forward, autograd, or split identity is wrong, return
to M25. Do not "fix" a mismatch by changing M25 from this mission.


## Frozen teaching fixtures

Declare the useful whole **before** the first injection.

| Fixture | Value |
| --- | --- |
| Loop | M25 `zero_grad → forward → loss → backward → step` |
| Device / dtype | CPU, `float64` |
| Train fixture | 36 clustered rows, splits 24 / 6 / 6, seed `2501` |
| Train net | `3→8→3` affine-ReLU-affine logits |
| Optimizer | SGD, momentum `0.9`, learning rate `0.25` |
| Eval | `model.eval()` + `no_grad`; never `step` |
| Held-out | disjoint; scored only after freeze |
| Injection rule | one named change per run |
| Hidden runs | omit defect, category, and knobs from public symptoms |

Primary sources: `pytorch-basics`, `fastai-course`, and
`karpathy-micrograd` in `data/source_registry.json`.


## Protocol (the whole we will reuse)

Symptom → competing hypotheses → cheapest discriminator → root cause →
smallest repair on the **broken object** → rerun original evidence →
regression.

A hypothesis is not a cause. A cause is not a repair. Architecture
changes are last, not first.


## Predict before running — known-good M25 trace

Timestamp a prediction before `run-healthy`.

Invariant: no M26 injection. Change: none. Predict train loss falls
across 8 epochs, held-out runs in eval mode, and every trainable
parameter moves. This trace is the useful whole later faults will be
judged against.


In [ ]:
healthy = known_good_m25_trace()
print("compact M25-style report")
print(compact_run_report(healthy.train_run))
print("symptoms", public_symptoms(healthy))
print("invariant failures", invariant_failures(healthy))
print("held-out training flag", healthy.held_out.model_training, "n", healthy.held_out.n)
print("layer moved", healthy.layer_moved)
assert healthy_invariants_hold(healthy)
assert m25_loop_order() == LOOP_ORDER


### A falling loss is a reference, not a diagnosis

This run is the known-good whole: same fixture, same loop, no injection.
Later cells change **one** control. If a later curve looks worse, the
comparison is this object, not a newly invented trainer.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
epochs = list(range(len(healthy.train_losses)))
ax.plot(epochs, healthy.train_losses, marker="o", label="train")
ax.plot(epochs, healthy.val_losses, marker="s", label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("mean softmax-NLL")
ax.set_title("Known-good M25 trace (held-out still frozen)")
ax.legend()
fig.tight_layout()
plt.show()
print("val is allowed during training; held-out is not on this figure")


## Predict before running — first hidden fault (symptoms only)

Timestamp a prediction before `run-hidden`.

You will see a public symptom report with **no category**. Write at
least four hypotheses spanning data, optimization, gradient flow,
architecture/regularization, and evaluation. Do not name a cause yet.
Predict which cheap measurement you would run first, and what outcome
would drop a family.


In [ ]:
hidden_payload = hidden_fault_run(HIDDEN_PRACTICE_SEED, reveal=False)
hidden_diag = hidden_payload["diag"]
hidden_fault = hidden_payload["fault"]
hidden_symptoms = hidden_payload["symptoms"]
print("hidden public symptoms")
for key, value in hidden_symptoms.items():
    print(f"  {key}: {value}")
assert "defect" not in hidden_symptoms
assert "category" not in hidden_symptoms
print("hidden repr", hidden_fault)
print("empty diagnosis record", empty_diagnosis_record())


### Symptoms are not a post-mortem

The public report is allowed to look "almost trained." Finite loss is
not health. Rank families before you open tensors, grads, or source.
Record the ranking in your log, not in this notebook.


## Predict before running — diagnostic battery

Timestamp a prediction before `run-hidden-disc`.

The battery is a **menu**: fixture-block label agreement, per-column
scales, layer movement and grads, tiny-subset overfit at current versus
restored knobs, honest versus claimed validation. Predict which one
result would raise your top hypothesis and which result would kill it.
Do not treat the whole menu as a substitute for a ranked guess.


In [ ]:
hidden_battery = diagnostic_battery(hidden_diag)
hidden_signals = signals_from_battery(hidden_battery, hidden_diag)
hidden_rank = rank_hypotheses(hidden_signals)
print("cluster agreement", hidden_battery["cluster_agreement"])
print("feature scales", hidden_battery["feature_scales"], "ratio", hidden_battery["feature_scale_ratio"])
print("layer moved", hidden_battery["layer_moved"])
print("grad report", hidden_battery["grad_report"])
print("tiny current", hidden_battery["tiny_overfit_current_knobs"]["overfit"], hidden_battery["tiny_overfit_current_knobs"]["final_train_loss"])
print("tiny restored knobs", hidden_battery["tiny_overfit_restored_knobs"]["overfit"], hidden_battery["tiny_overfit_restored_knobs"]["final_train_loss"])
print("claimed n/loss/train-flag", hidden_battery["claimed_n"], hidden_battery["claimed_val_loss"], hidden_battery["claimed_model_training"])
print("honest n/loss", hidden_battery["honest_n"], hidden_battery["honest_val_loss"])
print("positive-score family count", len(hidden_rank))
print("cheapest discriminator for current top", cheapest_discriminator(hidden_rank[0]))
print("invariant failures", invariant_failures(hidden_diag))


### Diagnose before repair

Update the ranking from the battery. A blocked path, a label scramble, a
bad learning rate, a tiny net, and a leaked split do **not** share a
single repair. If `fc1` never moved, widening the net is not smallest.
If labels disagree with fixture blocks, more Dropout is not smallest.

Leave the repair for `run-hidden-repair` after the named catalogue
calibrates the other families.


## Predict before running — data corruption

Timestamp a prediction before `run-data`.

Change: corrupt a bounded portion of **train** labels. Invariant:
architecture and optimizer fixed. Predict fixture-block agreement on
train versus val/held-out, and whether train loss can still fall.


In [ ]:
data_fault = prepare_fault("label_shuffle", seed=DEFAULT_SEED)
data_diag = run_prepared(data_fault)
print("agreement", cluster_label_agreement(data_fault.splits))
print("train losses", data_diag.train_losses)
print("honest val acc", data_diag.honest_val.accuracy)
print("failures", invariant_failures(data_diag))
print("symptoms (named)", {k: public_symptoms(data_diag)[k] for k in ("defect", "category", "final_train_loss", "final_reported_val_accuracy")})


### Train can look busy while labels lie

Val and held-out labels stayed honest. Agreement is a data check, not a
capacity check. The optimizer did not become the root cause just because
train loss moved.


## Predict before running — feature scaling

Timestamp a prediction before `run-scale`.

Change: multiply one feature column by a large constant on the shared
tensor. Invariant: architecture and optimizer fixed. Predict the
per-column standard-deviation ratio, and whether training stays stable.


In [ ]:
scale_fault = prepare_fault("feature_scale", seed=DEFAULT_SEED)
scale_diag = run_prepared(scale_fault)
print("feature scales", feature_column_scales(scale_fault.splits))
print("scale ratio", feature_scale_ratio(scale_fault.splits))
print("train losses (head, tail)", scale_diag.train_losses[0], scale_diag.train_losses[-1])
print("label agreement still", cluster_label_agreement(scale_fault.splits))
print("failures", invariant_failures(scale_diag))


### One wild column is not a new architecture

If column 0's scale dominates, SGD on an unnormalized feature is the
hypothesis to test. Fixture-block labels can still be honest. Do not
conflate explosion with "the net is too small."


## Predict before running — learning-rate failure

Timestamp a prediction before `run-lr`.

Change only the learning rate. Data, seed, architecture, and budget
stay fixed. First run a high rate, then a low rate. Predict oscillation
or divergence versus an almost-flat loss.


In [ ]:
lr_high_fault = prepare_fault("lr_high", seed=DEFAULT_SEED)
lr_high_diag = run_prepared(lr_high_fault)
lr_low_fault = prepare_fault("lr_low", seed=DEFAULT_SEED)
lr_low_diag = run_prepared(lr_low_fault)
print("high lr", lr_high_fault.learning_rate, "losses", lr_high_diag.train_losses)
print("high oscillates", public_symptoms(lr_high_diag)["train_loss_oscillates"])
print("low lr", lr_low_fault.learning_rate, "losses", lr_low_diag.train_losses)
print("low delta", lr_low_diag.train_losses[-1] - lr_low_diag.train_losses[0])
print("failures high", invariant_failures(lr_high_diag))
print("failures low", invariant_failures(lr_low_diag))


### Rate bugs are traces, not vibes

A high rate fights itself on this fixture. A low rate barely moves.
Neither one requires a new layer. Tiny-subset overfit at a restored
rate is the cheap check that the data pipeline still works.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(healthy.train_losses, marker="o", label="known-good lr=0.25")
ax.plot(lr_high_diag.train_losses, marker="s", label="high lr")
ax.plot(lr_low_diag.train_losses, marker="^", label="low lr")
ax.set_xlabel("epoch")
ax.set_ylabel("mean train softmax-NLL")
ax.set_title("Only the learning rate changed")
ax.legend()
fig.tight_layout()
plt.show()
print("same data, seed, width, and epoch budget")


## Predict before running — gradient-flow failure

Timestamp a prediction before `run-gradflow`.

Change: block the intended gradient path into `fc1`. Invariant: data
and forward contract fixed. Predict which parameters move, and whether
train loss can still fall through `fc2`.


In [ ]:
grad_fault = prepare_fault("frozen_layer", seed=DEFAULT_SEED)
grad_diag = run_prepared(grad_fault)
print("layer moved", grad_diag.layer_moved)
print("grad report", {k: grad_diag.grad_report[k] for k in ("fc1_grad_norm", "fc2_grad_norm", "fc1_requires_grad", "fc1_grad_is_none")})
print("train losses", grad_diag.train_losses)
print("failures", invariant_failures(grad_diag))


### A falling loss can hide a frozen layer

If `fc2` trains and `fc1` does not, the forward pass still runs. The
discriminator is per-parameter `requires_grad`, `.grad`, and movement —
not a global loss screenshot.


## Predict before running — capacity mismatch

Timestamp a prediction before `run-capacity`.

Change: hidden width `1` instead of `8`. Invariant: evaluation policy
fixed. Predict train and val both underfit relative to the known-good
trace, without claiming the data are corrupt.


In [ ]:
cap_fault = prepare_fault("tiny_hidden", seed=DEFAULT_SEED)
cap_diag = run_prepared(cap_fault)
print("n_hidden", cap_fault.n_hidden, "vs known-good", TRAIN_HIDDEN)
print("train losses", cap_diag.train_losses)
print("honest val acc", cap_diag.honest_val.accuracy, "known-good", healthy.honest_val.accuracy)
print("label agreement", cluster_label_agreement(cap_fault.splits))
print("failures", invariant_failures(cap_diag))


### Underfit is a comparison

Width 1 can still move parameters and still be too small for this
fixture's full train set. That is not an evaluation leak: claimed `n`
still matches val. Repair is restoring width on the same config, not
relabeling the data.


## Predict before running — crushing regularization

Timestamp a prediction before `run-reg`.

Change only Dropout (crushing `p`). Width, data, and optimizer family
stay fixed. Predict train loss stays high even if eval-mode metrics look
less noisy.


In [ ]:
reg_fault = prepare_fault("crushing_dropout", seed=DEFAULT_SEED)
reg_diag = run_prepared(reg_fault)
print("dropout_p", reg_fault.dropout_p)
print("train losses", reg_diag.train_losses)
print("honest val acc", reg_diag.honest_val.accuracy)
print("failures", invariant_failures(reg_diag))


### Regularization is a training-time tax

Crushing Dropout can prevent the train set from being fit. The smallest
repair is lowering Dropout on the same config, not inventing a new
backbone. Compare this underfit with the tiny-width underfit: same
family of "cannot fit," different object.


## Predict before running — evaluation defects

Timestamp a prediction before `run-eval`.

Invariant: the checkpoint is fixed. One run scores validation in train
mode (Dropout live). Another reports train rows as validation. Predict
whether parameters move, and which numbers can look better than held-out
without any training.


In [ ]:
mode_fault = prepare_fault("train_mode_eval", seed=DEFAULT_SEED)
mode_diag = run_prepared(mode_fault)
print("train-mode claimed", mode_diag.claimed_val.model_training, mode_diag.claimed_val.loss, mode_diag.claimed_val.accuracy)
print("train-mode honest", mode_diag.honest_val.model_training, mode_diag.honest_val.loss, mode_diag.honest_val.accuracy)
print("parameters updated during claimed eval", mode_diag.claimed_val.parameters_updated)

leak_fault = prepare_fault("val_leakage", seed=DEFAULT_SEED)
leak_diag = run_prepared(leak_fault)
print("leak claimed n/loss/acc", leak_diag.claimed_val.n, leak_diag.claimed_val.loss, leak_diag.claimed_val.accuracy)
print("leak honest n/loss/acc", leak_diag.honest_val.n, leak_diag.honest_val.loss, leak_diag.honest_val.accuracy)
print("held-out n", leak_diag.held_out.n)

mode_eval_repair = repair_and_verify(mode_diag)
print("mode repair retrained", mode_eval_repair["retrained"], "params unchanged", mode_eval_repair["parameters_unchanged"])
print("mode repaired training flag", mode_eval_repair["claimed"].model_training)

leak_eval_repair = repair_and_verify(leak_diag)
print("leak repair retrained", leak_eval_repair["retrained"], "claimed n", leak_eval_repair["claimed"].n)


### Metrics can be fiction on a frozen checkpoint

Train-mode evaluation can change outputs without `optimizer.step`.
Scoring train rows as val can copy the train metric. The smallest
repair is the evaluation call on **this** checkpoint. Retraining to
dodge a reporting bug is not a repair.


## Predict before running — smallest repair of the hidden fault

Timestamp a prediction before `run-hidden-repair`.

Apply `repair_and_verify` to the **same** prepared objects from
`run-hidden`. Predict that a training-time fault will retrain on those
objects, an evaluation fault will not, and healthy invariants return
without building a separate `defect="none"` twin as the "fix."


In [ ]:
hidden_repair = repair_and_verify(hidden_diag)
print("retrained", hidden_repair["retrained"], "same splits", hidden_repair["same_splits"])
print("repair failures", hidden_repair["failures"])
print("repaired symptoms finite", public_symptoms(hidden_repair["diag"])["finite_train_losses"])
assert hidden_repair["failures"] == ()


## Code reading — injection harness beside the clean loop

Read the M26 harness next to M25 `train_model`. Predict the
object each family mutates **before** `run-code-reading`.


## Predict before running — code reading

Timestamp a prediction before `run-code-reading`.

Predict:

- which object `label_shuffle` mutates, and which splits stay honest
- how a frozen `fc1` can still produce a falling train loss
- why `claimed_eval` can look better than held-out without training
- what `public_symptoms` must omit on a hidden run
- why repairing evaluation on the same checkpoint is smaller than
  changing width


In [ ]:
print(inspect.getsource(prepare_fault))
print("---")
print(inspect.getsource(apply_model_hooks))
print("---")
print(inspect.getsource(claimed_eval))
print("---")
print(inspect.getsource(repair_prepared))
print("---")
print(inspect.getsource(public_symptoms))
print("M25 runner module", train_model.__module__)
print("M26 does not define train_model:", "train_model" not in globals() or train_model.__module__.endswith("pytorch_training"))


## Predict before running — Controlled failure: Chaos Day

Timestamp a prediction before `run-chaos`.

Phase-end hidden fault. Public symptoms only. Write a fresh ranked
hypothesis list across the same families. Do not reuse the practice
fault's answer by habit. Do not name a cause from the seed.


In [ ]:
chaos_payload = chaos_day(CHAOS_SEED, reveal=False)
chaos_diag = chaos_payload["diag"]
chaos_fault = chaos_payload["fault"]
chaos_symptoms = chaos_payload["symptoms"]
print("chaos public symptoms")
for key, value in chaos_symptoms.items():
    print(f"  {key}: {value}")
assert "defect" not in chaos_symptoms
assert "category" not in chaos_payload
print("chaos repr", chaos_fault)


### Chaos Day is a release gate, not a puzzle caption

V05 does not ship a training change because a dashboard turned green.
It ships when a hidden defect has ranking, a discriminator, a smallest
repair, and a regression check.


## Predict before running — Chaos Day discriminator

Timestamp a prediction before `run-chaos-disc`.

Run the same cheap menu. Predict how you will update the ranking, and
which single outcome would be enough to stop looking.


In [ ]:
chaos_battery = diagnostic_battery(chaos_diag)
chaos_signals = signals_from_battery(chaos_battery, chaos_diag)
chaos_rank = rank_hypotheses(chaos_signals)
print("cluster agreement", chaos_battery["cluster_agreement"])
print("feature scale ratio", chaos_battery["feature_scale_ratio"])
print("layer moved", chaos_battery["layer_moved"])
print("grad report", chaos_battery["grad_report"])
print("tiny current overfit", chaos_battery["tiny_overfit_current_knobs"]["overfit"], chaos_battery["tiny_overfit_current_knobs"]["final_train_loss"])
print("tiny restored overfit", chaos_battery["tiny_overfit_restored_knobs"]["overfit"])
print("claimed n vs honest n", chaos_battery["claimed_n"], chaos_battery["honest_n"])
print("positive-score family count", len(chaos_rank))
print("cheapest discriminator for current top", cheapest_discriminator(chaos_rank[0]))
print("invariant failures", invariant_failures(chaos_diag))


## Predict before running — Chaos Day smallest repair

Timestamp a prediction before `run-chaos-repair`.

Repair the prepared Chaos Day objects. Predict original evidence
returns to healthy invariants, and that you did not construct a second
independent healthy run as the repair.


In [ ]:
chaos_repair = repair_and_verify(chaos_diag)
print("retrained", chaos_repair["retrained"], "same splits", chaos_repair["same_splits"])
print("repair failures", chaos_repair["failures"])
assert chaos_repair["failures"] == ()
print("repaired train losses", chaos_repair["diag"].train_losses)


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- the known-good M25 trace
- ranked hypotheses for the hidden fault and for Chaos Day
- named catalogue traces (labels, scale, learning rate, frozen path,
  capacity, Dropout, evaluation)
- smallest repairs driven from broken objects
- regression evidence

See `missions/M26/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M26/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams. Use a
**fresh seed**.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Complete `missions/M26/adr_prompt.md` using `templates/ADR.md`. The
decision is the V05 failure-triage/release policy: sanity checks,
escalation order, rollback, and evidence required before architecture
changes.

**Status:** [UNFILLED BY LEARNER]


## M25 → M26 → P5 handoff

M25 wrapped reverse-mode numbers in an instrumented loop. M26 taught
diagnosis on that loop: one control at a time, cheapest discriminator,
smallest repair, regression.

P5 inherits the protocol. Language-model internals stay closed here.
M26 does not teach them.


## Mission summary prompt

In your own words, using only numbers from this lab:

1. Why is a falling train loss not a diagnosis?
2. Which discriminator separates corrupted train labels from a tiny net?
3. How can `fc2` training hide a frozen `fc1`?
4. Why is repairing evaluation on a frozen checkpoint smaller than
   retraining?
5. What must P5 receive that a validation screenshot cannot provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert healthy_invariants_hold(healthy)
assert "defect" not in hidden_symptoms
assert cluster_label_agreement(data_fault.splits)["train_agreement"] < 0.85
assert feature_scale_ratio(scale_fault.splits) >= 10
assert public_symptoms(lr_high_diag)["train_loss_oscillates"]
assert abs(lr_low_diag.train_losses[-1] - lr_low_diag.train_losses[0]) < 0.05
assert not grad_diag.layer_moved["fc1.weight"]
assert cap_diag.train_losses[-1] > 0.25
assert mode_diag.claimed_val.model_training
assert leak_diag.claimed_val.n != leak_diag.honest_val.n
assert hidden_repair["failures"] == ()
assert "defect" not in chaos_symptoms
assert chaos_repair["failures"] == ()
assert m25_loop_order() == LOOP_ORDER
print("M26 integrity checks passed")
